# Machine Learning Basics: A Hands-On Practical
### IndabaX — Deep Learning Indaba

**Duration:** ~30 minutes
**Level:** Beginner (no prior ML experience assumed)

**By the end of this session you will be able to:**
- Describe the essential steps in building any machine learning model
- Build, train, and evaluate a simple classifier in scikit-learn
- Recognise overfitting vs. underfitting
- Start using **Kaggle** to find datasets, practice, and compete

**Agenda**
1. What is Machine Learning? *(2 min)*
2. The ML workflow *(3 min)*
3. Hands-on: build a model step-by-step *(18 min)*
4. Using Kaggle *(5 min)*
5. Wrap-up & next steps *(2 min)*


## 1. What is Machine Learning?

Instead of writing explicit rules for every situation, we show a computer many **examples** and let it learn the underlying pattern.

**Main types of ML:**
- **Supervised learning** — learn from labelled examples (inputs + correct answers). *e.g. is this email spam?*
- **Unsupervised learning** — find structure in unlabelled data. *e.g. group customers into segments*
- **Reinforcement learning** — learn by trial and error from rewards. *e.g. an agent playing a game*

Today we'll focus on **supervised learning — classification**, since it's the clearest way to see every step of the workflow. Everything here (train/test split, evaluation, overfitting) applies just as much when your "model" is a deep neural network instead of the simple models we use today.


## 2. The Machine Learning Workflow

No matter how simple or advanced the model, almost every ML project follows the same backbone:

1. **Define the problem** — what are you predicting, and why?
2. **Get the data** — collect or download a relevant dataset
3. **Explore & clean the data (EDA)** — understand it, fix issues
4. **Split the data** — separate data the model learns from vs. data used to check it
5. **Choose & train a model** — start simple, then try more complex options
6. **Evaluate the model** — measure performance on unseen data
7. **Iterate & improve** — tune, engineer features, try other models
8. **Share / deploy** — put it to use

We'll now work through steps 2–7 hands-on.


## 3. Setup

We use `scikit-learn`, a beginner-friendly Python library that covers most of what you need for classical ML (data splitting, models, evaluation).

In [ ]:
# NumPy - for numerical arrays and math operations
import numpy as np
# pandas - for loading and working with tabular data (DataFrames)
import pandas as pd
# matplotlib's plotting interface - for creating charts
import matplotlib.pyplot as plt
# seaborn - built on matplotlib, makes statistical plots easier and nicer-looking
import seaborn as sns

# load_breast_cancer - a small, built-in classification dataset we'll use today
from sklearn.datasets import load_breast_cancer
# train_test_split - splits data into training and testing sets
# cross_val_score - evaluates a model using k-fold cross-validation
from sklearn.model_selection import train_test_split, cross_val_score
# StandardScaler - rescales features to have mean 0 and standard deviation 1
from sklearn.preprocessing import StandardScaler
# LogisticRegression - a simple, interpretable classification model
from sklearn.linear_model import LogisticRegression
# DecisionTreeClassifier - a tree-based classification model
from sklearn.tree import DecisionTreeClassifier
# confusion_matrix - table of correct/incorrect predictions per class
# classification_report - precision/recall/F1 summary per class
# ConfusionMatrixDisplay - draws a confusion matrix as a chart
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Jupyter "magic" command: display matplotlib plots directly inside the notebook
%matplotlib inline
# Use seaborn's "whitegrid" theme for every plot we draw from here on
sns.set_style("whitegrid")

## 4. Get the Data

For this live practical we use a dataset that ships **inside** scikit-learn, so everyone can follow along instantly with no downloads or internet dependency.

**Task:** predict whether a breast tumour is **malignant** or **benign** from 30 measurements taken from a scan. This is a binary classification problem.

> In the real world you'd usually load data from a CSV, a database, or a site like **Kaggle** (more on that in Section 8) — the steps that follow are identical either way.


In [ ]:
# Load the built-in breast cancer dataset as a scikit-learn "Bunch" object
data = load_breast_cancer()
# Convert the raw feature array into a labelled pandas DataFrame (adds column names)
df = pd.DataFrame(data.data, columns=data.feature_names)
# Add the target labels as a new column (0 = malignant, 1 = benign)
df['target'] = data.target  # 0 = malignant, 1 = benign

# Print the DataFrame's dimensions as (rows, columns)
print(f"Shape: {df.shape}")
# Display the first 5 rows so we can eyeball what the data looks like
df.head()

## 5. Explore the Data (EDA)

Before touching any model, always ask:
- How much data do I have?
- What do the features look like (scale, type)?
- Is the target balanced?
- Are there missing values?


In [ ]:
# Show column names, data types, and non-null (non-missing) counts for every column
df.info()

In [ ]:
# Count how many rows belong to each class (0 = malignant, 1 = benign)
df['target'].value_counts()

In [ ]:
# Draw a bar chart showing how many samples fall into each class
sns.countplot(x='target', data=df)
# Add a descriptive title above the chart
plt.title('Class distribution (0 = malignant, 1 = benign)')
# Label the x-axis so the classes are clear
plt.xlabel('Class')
# Render the chart in the notebook
plt.show()

In [ ]:
# Check for missing values:
# df.isnull() -> True/False for every single cell in the DataFrame
# .sum()      -> counts the missing values in each column
# .sum()      -> adds those column totals into one grand total
df.isnull().sum().sum()

*Real-world data (including most Kaggle datasets) is rarely this clean — expect missing values, wrong types, and categorical columns that need encoding.*

## 6. Prepare the Data

Two key ideas:
- **Train/test split** — we hold back some data the model never sees during training, so we can honestly check how well it generalises.
- **Feature scaling** — many algorithms (like logistic regression) perform better when features are on a similar scale.


In [ ]:
# X holds all the input features (every column except the target)
X = df.drop(columns='target')
# y holds only the target column - the thing we want to predict
y = df['target']

# Randomly split the data into a training set (80%) and a test set (20%)
# random_state=42 makes the split reproducible (same split every time we run it)
# stratify=y keeps the same class balance in both the train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Confirm how many rows ended up in each split
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Create a scaler that will standardise features to mean 0, standard deviation 1
scaler = StandardScaler()
# Learn the scaling parameters (mean/std) from the TRAINING data, then apply them
X_train_scaled = scaler.fit_transform(X_train)
# Apply that SAME learned scaling to the test data (never re-fit on test data)
X_test_scaled = scaler.transform(X_test)

**Important:** we `fit` the scaler only on the training data, then `transform` the test data with those same parameters. Fitting on the test set too would leak information from data the model is supposed to have never seen.

## 7. Choose & Train a Model

Good practice: start with a simple, interpretable **baseline** model before reaching for anything fancier.


In [ ]:
# Create a Logistic Regression model; max_iter is raised so it has room to converge
log_reg = LogisticRegression(max_iter=5000)
# Train ("fit") the model using the scaled training features and their true labels
log_reg.fit(X_train_scaled, y_train)

# Accuracy of the model on the data it was trained on
train_acc = log_reg.score(X_train_scaled, y_train)
# Accuracy of the model on data it has never seen before (the test set)
test_acc = log_reg.score(X_test_scaled, y_test)

# Print both accuracies so we can compare them
print(f"Logistic Regression — Train accuracy: {train_acc:.3f}, "
      f"Test accuracy: {test_acc:.3f}")

In [ ]:
# Create a Decision Tree model; max_depth=4 limits how complex/deep it can grow
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
# Train the tree on the (unscaled) training data - trees don't need feature scaling
tree.fit(X_train, y_train)

# Accuracy on the training data
train_acc_tree = tree.score(X_train, y_train)
# Accuracy on the held-out test data
test_acc_tree = tree.score(X_test, y_test)

# Print both accuracies for comparison
print(f"Decision Tree — Train accuracy: {train_acc_tree:.3f}, "
      f"Test accuracy: {test_acc_tree:.3f}")

## 8. Evaluate the Model

Accuracy alone can be misleading (especially with imbalanced classes). A **confusion matrix** and per-class metrics give a fuller picture.


In [ ]:
# Use the trained logistic regression model to predict labels for the test set
y_pred = log_reg.predict(X_test_scaled)

# Build a confusion matrix: rows = actual class, columns = predicted class
cm = confusion_matrix(y_test, y_pred)
# Draw the confusion matrix as a labelled, colour-coded chart
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Blues')
# Add a title to the chart
plt.title('Logistic Regression — Confusion Matrix')
# Render the chart
plt.show()

# Print precision, recall, F1-score, and support (sample count) for each class
print(classification_report(y_test, y_pred, target_names=data.target_names))

### Overfitting vs. Underfitting

- **Overfitting**: the model memorises the training data — high train accuracy, much lower test accuracy.
- **Underfitting**: the model is too simple to capture the pattern — low accuracy on *both* train and test.

Let's see overfitting in action by removing the depth limit on our tree:


In [ ]:
# Create a Decision Tree with NO depth limit - free to grow as complex as it wants
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=42)  # no depth limit
# Train the unrestricted tree on the training data
deep_tree.fit(X_train, y_train)

# Print train vs. test accuracy - watch for a big gap, which signals overfitting
print(f"Unrestricted tree — Train accuracy: {deep_tree.score(X_train, y_train):.3f}, "
      f"Test accuracy: {deep_tree.score(X_test, y_test):.3f}")

Notice train accuracy climbs close to 1.0 while test accuracy doesn't improve (or drops) — a classic overfitting signature. Fixes include: simpler models, more data, regularisation, or techniques like cross-validation to get a more reliable performance estimate.

## 9. Iterate & Improve

Once you have a working baseline, common next steps are:
- Try other algorithms (Random Forest, Gradient Boosting, a small neural network, ...)
- Tune hyperparameters (e.g. with `GridSearchCV`)
- Engineer better features
- Get more or better-quality data
- Use **cross-validation** for a more robust performance estimate than a single train/test split


In [ ]:
# Run 5-fold cross-validation: the data is split 5 different ways, and the model
# is trained and tested 5 times so we get 5 accuracy scores instead of just one
cv_scores = cross_val_score(
    LogisticRegression(max_iter=5000), scaler.fit_transform(X), y, cv=5
)
# Print the average accuracy and its spread (standard deviation) across the 5 folds
print(f"5-fold CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 10. Using Kaggle

**Kaggle** (kaggle.com) is one of the best places to practice everything above on real, messy data.

**What's on Kaggle:**
- **Datasets** — tens of thousands of free datasets on almost any topic, ready to practice the workflow above
- **Competitions** — predict on real problems, submit results, and see how you rank on a public leaderboard
- **Notebooks** — free cloud Jupyter notebooks with GPU/TPU access, no local install needed — great if your laptop can't run heavy models
- **Kaggle Learn** — short, free, practical micro-courses (e.g. "Intro to Machine Learning")

**Getting started (do this after the session):**
1. Create a free account at kaggle.com
2. Open the **"Titanic – Machine Learning from Disaster"** competition — the classic first project
3. Click **"New Notebook"** on the competition page — this gives you a free, ready-to-run notebook already connected to the data
4. Apply the exact same workflow from today: load → explore → clean → split → train → evaluate → submit

If you have the Kaggle API set up locally, you can also pull data directly into a notebook like this one:


In [ ]:
# Optional — only if you have a kaggle.json API token configured locally:
# !pip install kaggle
# !kaggle competitions download -c titanic

## 11. Wrap-Up

**The essentials, recapped:**
1. Define the problem
2. Get the data
3. Explore & clean it
4. Split into train/test
5. Train a simple model first
6. Evaluate honestly (beyond just accuracy)
7. Watch for overfitting/underfitting
8. Iterate

**Next steps:**
- Repeat this exact notebook on the Kaggle Titanic dataset
- Try Kaggle Learn's "Intro to Machine Learning" course
- Look for African-context datasets/competitions (e.g. on Zindi) to practice on problems relevant to home


## 12. Try It Yourself 🎯

Pick one:
- Swap the dataset for `sklearn.datasets.load_wine()` or `load_iris()` and repeat steps 4–8
- Download the Kaggle Titanic dataset and repeat the same 8-step workflow on it

Use the empty cell below to get started.
